In [4]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.utils import resample
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
import cupy
import os

kaggle = True if os.environ.get('KAGGLE_URL_BASE','') else False
balanced = True

if kaggle:
    training_data = '/kaggle/input/competitions/playground-series-s6e4/train.csv'
else:
    training_data = 'data/train.csv'

df_tv = pd.read_csv(training_data)
counts = df_tv['Irrigation_Need'].value_counts()
continous_variables = df_tv.select_dtypes(['float64']).columns

#balanced data set
df_tv_majority = df_tv[df_tv['Irrigation_Need'] == counts.keys()[0]]
df_tv_mid = df_tv[df_tv['Irrigation_Need'] == counts.keys()[1]]
df_tv_minority = df_tv[df_tv['Irrigation_Need'] == counts.keys()[2]]

# Downsample majority class
majority_downsampled = resample(df_tv_majority, 
                              replace=False,  # Sample without replacement
                              n_samples=len(df_tv_mid),  # Equalize class sizes
                              random_state=42)
# Oversample minority class
minority_upsampled = resample(df_tv_minority, 
                              replace=True,  # Sample with replacement
                              n_samples=len(df_tv_mid),  # Equalize class sizes
                              random_state=42)
# # Oversample mid class
# mid_upsampled = resample(df_tv_mid, 
#                          replace=True,  # Sample with replacement
#                          n_samples=len(df_tv_majority),  # Equalize class sizes
#                          random_state=42)
#df_balanced = pd.concat([df_tv_majority, mid_upsampled, minority_upsampled])
df_balanced = pd.concat([df_tv_mid, majority_downsampled, minority_upsampled])    
df_tv = df_balanced if balanced else df_tv



class_le = LabelEncoder()
y = class_le.fit_transform(df_balanced['Irrigation_Need'].values)

df_dummy = pd.get_dummies(df_balanced.iloc[:,1:-1], dtype=int)

sc = StandardScaler().fit(df_dummy[continous_variables])
df_dummy[continous_variables] = sc.transform(df_dummy[continous_variables])
x = df_dummy.to_numpy()

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = \
    train_test_split(x, y, 
                     test_size=0.20,
                     stratify=y,
                     random_state=1)


# mxgb_gs = GridSearchCV(xgb.XGBClassifier(random_state=1, tree_method='hist', device='cuda', n_jobs=-1, objective='multi:softmax', num_class=3), 
#                          param_grid={'n_estimators': [100, 500, 1000], 'learning_rate': [0.01, 0.1], 
#                                      'max_depth': [3, 4, 5, 10], 'lambda': [1, 10, 100]}, 
#                          cv=5, scoring='balanced_accuracy')

# mxgb_gs.fit(cupy.array(X_train), y_train)
# mxgb_gs.score(X_test, y_test)
mxgb = xgb.XGBClassifier(random_state=1, tree_method='hist', device='cuda', n_jobs=-1,
                        n_estimators=1000, learning_rate=0.1, max_depth=10, reg_lambda=10, objective='multi:softmax', num_class=3)
#mxgb.fit(cupy.array(X_train), y_train)
#mxgb.score(X_test, y_test)
mxgb.fit(cupy.array(x), y)


,objective,'multi:softmax'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,'cuda'
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [ ]:
if kaggle:
    testing_data = '/kaggle/input/competitions/playground-series-s6e4/test.csv'
else:
    testing_data = 'data/test.csv'

df_test = pd.read_csv(testing_data)

ids = df_test['id'].values

df_test_dummy = pd.get_dummies(df_test.iloc[:,1:])
df_test_dummy[continous_variables] = sc.transform(df_test_dummy[continous_variables])
x_test = df_test_dummy.to_numpy()

if kaggle:
    out_dir = '/kaggle/working/'
else:
    out_dir = 'data/'

df_submission_mxgb = pd.DataFrame({'id': ids, 'Irrigation_Need': class_le.inverse_transform(mxgb.predict(x_test))})
df_submission_mxgb.to_csv(os.path.join(out_dir, 'submission-mxgb_v1.csv'), index=False)

OSError: Cannot save file into a non-existent directory: '/kaggle/working'